In [ ]:
import pandas as pd
import plotly.express as px
from math import pi, sqrt, pow

# 1 - Càlcul de les àrees del gràfic original

In [2]:
df = pd.read_csv("../data/global_births_badgraph_data.csv")
df["Area(px^2)"] = df["Horizontal(px)"] * df["Vertical(px)"] * pi

In [3]:
df["Radius(px)"] = df["Area(px^2)"].apply(lambda x: sqrt(x))
df = df[["Continent", "Births(M)", "Radius(px)"]]
df

,Continent,Births(M),Radius(px)
0,Asia,64.9,782.170449
1,Africa,47.6,586.510845
2,Latam & Caribbean,9.3,345.173726
3,Europe,6.1,330.295241
4,North America,4.0,304.346378
5,Oceania,0.7,226.763287
6,Antartica,0.0,180.712079


# 2 - Tractament de la font de les dades

In [4]:
un_wpp24_df = pd.read_csv("../data/UN_PPP2024_Output_Births.csv")
un_wpp24_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 87 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            302 non-null    int64  
 1   Variant          302 non-null    str    
 2   Region           302 non-null    str    
 3   Notes            79 non-null     str    
 4   Location code    302 non-null    int64  
 5   ISO3 Alpha-code  237 non-null    str    
 6   ISO2 Alpha-code  236 non-null    str    
 7   SDMX code**      283 non-null    float64
 8   Type             302 non-null    str    
 9   Parent code      302 non-null    int64  
 10  2024             302 non-null    str    
 11  2025             302 non-null    str    
 12  2026             302 non-null    str    
 13  2027             302 non-null    str    
 14  2028             302 non-null    str    
 15  2029             302 non-null    str    
 16  2030             302 non-null    str    
 17  2031             302 non-nu

In [5]:
un_wpp24_df = un_wpp24_df[["Region", "Location code", "Type", "Parent code", "2026"]]
un_wpp24_df.info()
un_wpp24_df

<class 'pandas.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Region         302 non-null    str  
 1   Location code  302 non-null    int64
 2   Type           302 non-null    str  
 3   Parent code    302 non-null    int64
 4   2026           302 non-null    str  
dtypes: int64(2), str(3)
memory usage: 11.9 KB


,Region,Location code,Type,Parent code,2026
0,World,900,World,0,132 503
1,Sustainable Development Goal (SDG) regions,1828,Label/Separator,900,...
2,Sub-Saharan Africa,1834,SDG region,1828,41 678
3,Northern Africa and Western Asia,1833,SDG region,1828,11 881
4,Central and Southern Asia,1831,SDG region,1828,38 672
...,...,...,...,...,...
297,Samoa,882,Country/Area,957,5
298,Tokelau,772,Country/Area,957,0
299,Tonga,776,Country/Area,957,2
300,Tuvalu,798,Country/Area,957,0


In [6]:
un_wpp24_df["Births in 2026 (thousands)"] = un_wpp24_df[un_wpp24_df["2026"] != "..."]["2026"].apply(lambda x: int(x.replace(" ", "")))
un_wpp24_df.drop("2026", axis=1, inplace=True)
un_wpp24_df

,Region,Location code,Type,Parent code,Births in 2026 (thousands)
0,World,900,World,0,132503.0
1,Sustainable Development Goal (SDG) regions,1828,Label/Separator,900,NaN
2,Sub-Saharan Africa,1834,SDG region,1828,41678.0
3,Northern Africa and Western Asia,1833,SDG region,1828,11881.0
4,Central and Southern Asia,1831,SDG region,1828,38672.0
...,...,...,...,...,...
297,Samoa,882,Country/Area,957,5.0
298,Tokelau,772,Country/Area,957,0.0
299,Tonga,776,Country/Area,957,2.0
300,Tuvalu,798,Country/Area,957,0.0


In [7]:
treated_df = un_wpp24_df[un_wpp24_df["Type"] == "Region"]
treated_df

,Region,Location code,Type,Parent code,Births in 2026 (thousands)
37,Africa,903,Region,1840,47562.0
101,Asia,935,Region,1840,64911.0
158,Europe,908,Region,1840,6141.0
213,Americas,5505,Region,1840,13185.0
214,Latin America and the Caribbean,904,Region,1840,9160.0
268,Northern America,905,Region,1840,4036.0
274,Oceania,909,Region,1840,693.0


In [8]:
source_df = treated_df[["Region", "Births in 2026 (thousands)"]]
source_df

,Region,Births in 2026 (thousands)
37,Africa,47562.0
101,Asia,64911.0
158,Europe,6141.0
213,Americas,13185.0
214,Latin America and the Caribbean,9160.0
268,Northern America,4036.0
274,Oceania,693.0


In [9]:
source_df = source_df[source_df["Region"] != "Americas"]

# 3 - Unió de datasets (àrea bombolles + font oficial)

In [10]:
source_df

,Region,Births in 2026 (thousands)
37,Africa,47562.0
101,Asia,64911.0
158,Europe,6141.0
214,Latin America and the Caribbean,9160.0
268,Northern America,4036.0
274,Oceania,693.0


In [11]:
df = df.drop("Births(M)", axis=1)
df

,Continent,Radius(px)
0,Asia,782.170449
1,Africa,586.510845
2,Latam & Caribbean,345.173726
3,Europe,330.295241
4,North America,304.346378
5,Oceania,226.763287
6,Antartica,180.712079


In [12]:
change_name = {
    214: "Latam & Caribbean",
    268: "North America"
               }

for k, v in change_name.items():
    source_df.loc[k, "Region"] = v

In [13]:
source_df.rename({"Region": "Continent"}, inplace=True, axis=1)

In [14]:
source_df

,Continent,Births in 2026 (thousands)
37,Africa,47562.0
101,Asia,64911.0
158,Europe,6141.0
214,Latam & Caribbean,9160.0
268,North America,4036.0
274,Oceania,693.0


In [15]:
final_df = df.merge(source_df, on="Continent")
final_df

,Continent,Radius(px),Births in 2026 (thousands)
0,Asia,782.170449,64911.0
1,Africa,586.510845,47562.0
2,Latam & Caribbean,345.173726,9160.0
3,Europe,330.295241,6141.0
4,North America,304.346378,4036.0
5,Oceania,226.763287,693.0


In [ ]:
final_df.to_csv("../data/global_births_full_data.csv", index=False)

# 4 - Graph

In [2]:
df = pd.read_csv("../data/global_births_full_data.csv")
fig = px.bar(df, x='Continent', y='Births in 2026 (thousands)')
fig.show()

In [ ]:
bubble_df = df.copy()
bubble_df['y'] = 1
bubble_df['chart_size'] = pi * pow(bubble_df['Radius(px)'], 2)
chart_fig = px.scatter(bubble_df, x='Continent', y='y', size='Births in 2026 (thousands)', size_max=80)
chart_fig.show()

real_fig = px.scatter(bubble_df, x='Continent', y='y', size='chart_size', size_max=80)
real_fig.show()